# 10 · BlazeFace (short-range)

> **Placeholder:** no code yet. Filled in during **Phases 5–6** of `docs/EXPERIMENT_PLAN.md`.

## Purpose
Train, quantize, deploy and measure this detector on the ESP32-S3 (TFLite Micro + ESP-NN), against the ESP-DL baseline B0.

## Experiments covered
- P1 (PTQ int8 per-channel)
- Q1 (QAT int8 per-channel)

## Inputs
- Datasets from `01_datasets.ipynb` (A-train, A-test, B-cal, B-test, negatives)
- Input size: 128×128 (resized on device from the 320×240 frame)
- Toolchain validated in `00_toolchain_and_fidelity.ipynb`

## Outputs
- `artifacts/<model>/` float checkpoint, PTQ and QAT `.tflite` files
- `results/` JSONL + summary files from `tools/bench/run_bench.py`
- Rows for the metrics table in `docs/EXPERIMENT_PLAN.md` §7

## Depends on
- Phase 3 toolchain + fidelity gate passed
- Phase 4 TFLite Micro backend on the device
- Phase 1 datasets

## Model card
- Origin: MediaPipe face detection, short-range variant (TFLite native; PyTorch ports exist)
- Input 128×128; single-shot detector with fixed anchors
- Very low compute; depthwise-separable blocks

## Hypotheses (written before running)
- Fastest candidate; faster than B0
- Lower recall than B0 on S-bucket faces because of the 128×128 input
- Large PTQ → QAT gain (depthwise layers, GH2)

## 1. Obtain the float model

Get pretrained weights (or port the reference implementation to PyTorch). Record source, licence, parameter count and FLOPs at the planned input size.

## 2. Float reference (T1)

Evaluate the float32 model on the PC on A-test and B-test (AP50, recall by size). This is the reference every quantized variant's ΔAP is measured against.

## 3. Export check

PyTorch → PT2E → `ai-edge-torch` (fallback ONNX → `onnx2tf`) → `.tflite`. List every operator and check it against TFLite Micro + ESP-NN support. If it can't be deployed, record that and the reason as the result.

## 4. Device pre/post-processing spec

Input size, colour order and normalisation; output tensors; anchor/prior decoding; NMS settings. This becomes the model's detection-head metadata for the TFLM backend (Phase 4).

## 5. PTQ int8 per-channel (P row)

Calibrate on A-train (≤ 500 images) + B-cal. Full-integer int8, per-channel weights. Evaluate on the PC first.

## 6. Deploy PTQ to the S3

`tools/models/pack_models.py` → `flash_models.py` → `tools/bench/run_bench.py` on A-test and B-test (preview off). Record latency (median/p95/p99) and memory.

## 7. QAT int8 per-channel (Q row)

Fine-tune from the float weights with fake-quant inserted (PT2E `prepare_qat_pt2e`). Fixed seed, epochs and learning-rate schedule recorded here. Export and evaluate on the PC.

## 8. Deploy QAT to the S3

Same as the PTQ deployment. Latency should match the PTQ model exactly (GH1); only accuracy should change.

## 9. Results and conclusion

Fill this model's rows of the §7 metrics table in `docs/EXPERIMENT_PLAN.md` and state whether each hypothesis held.